In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

class RBM(nn.Module):
    def __init__(self, visible_dim, hidden_dim):
        super(RBM, self).__init__()
        self.W = nn.Parameter(torch.randn(visible_dim, hidden_dim) * 0.1)
        self.h_bias = nn.Parameter(torch.zeros(hidden_dim))
        self.v_bias = nn.Parameter(torch.zeros(visible_dim))

    def sample_hidden(self, v):
        h_prob = torch.sigmoid(torch.matmul(v, self.W) + self.h_bias)
        h_sample = torch.bernoulli(h_prob)
        return h_prob, h_sample

    def sample_visible(self, h):
        v_prob = torch.sigmoid(torch.matmul(h, self.W.t()) + self.v_bias)
        v_sample = torch.bernoulli(v_prob)
        return v_prob, v_sample

    def contrastive_divergence(self, v_input, learning_rate=0.01):
        # Pha Dương
        h_prob_pos, h_sample_pos = self.sample_hidden(v_input)
        pos_associations = torch.matmul(v_input.t(), h_prob_pos)

        # Pha Âm
        v_prob_neg, _ = self.sample_visible(h_sample_pos)
        h_prob_neg, _ = self.sample_hidden(v_prob_neg)
        neg_associations = torch.matmul(v_prob_neg.t(), h_prob_neg)

        # Cập nhật trọng số
        num_samples = v_input.size(0)
        self.W.data += learning_rate * (pos_associations - neg_associations) / num_samples
        self.v_bias.data += learning_rate * torch.sum(v_input - v_prob_neg, dim=0) / num_samples
        self.h_bias.data += learning_rate * torch.sum(h_prob_pos - h_prob_neg, dim=0) / num_samples

        error = torch.mean((v_input - v_prob_neg) ** 2)
        return error

class DBN(nn.Module):
    def __init__(self, layer_sizes, num_classes):
        super(DBN, self).__init__()
        self.rbm_layers = nn.ModuleList()
        for i in range(len(layer_sizes) - 1):
            self.rbm_layers.append(RBM(layer_sizes[i], layer_sizes[i+1]))
        self.classifier = nn.Linear(layer_sizes[-1], num_classes)

    def pretrain(self, dataloader, epochs=5, learning_rate=0.01):
        print("Bắt đầu Tiền huấn luyện (Pre-training) với dữ liệu MNIST...")
        for i, rbm in enumerate(self.rbm_layers):
            print(f"  -> Đang huấn luyện RBM tầng {i+1}...")
            for epoch in range(epochs):
                epoch_error = 0
                for batch_idx, (data, _) in enumerate(dataloader):
                    input_data = data
                    # Truyền dữ liệu qua các tầng RBM trước đó (nếu có) để lấy đặc trưng tiềm ẩn
                    with torch.no_grad():
                        for j in range(i):
                            _, input_data = self.rbm_layers[j].sample_hidden(input_data)

                    # Cập nhật RBM hiện tại
                    error = rbm.contrastive_divergence(input_data, learning_rate)
                    epoch_error += error.item()

                print(f"     Epoch {epoch+1}/{epochs} - Sai số tái tạo trung bình: {epoch_error/len(dataloader):.4f}")

    def forward(self, x):
        for rbm in self.rbm_layers:
            x = torch.sigmoid(torch.matmul(x, rbm.W) + rbm.h_bias)
        return self.classifier(x)


if __name__ == "__main__":
    # Data prep
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(lambda x: torch.flatten(x))
    ])

    print("Đang tải dữ liệu MNIST...")
    train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    # Chia dữ liệu thành các batch 64 ảnh/lần cập nhật
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

    # Khởi tạo DBN
    dbn = DBN(layer_sizes=[784, 256, 128], num_classes=10)

    # Tiền huấn luyện (Chỉ dùng ảnh, không dùng nhãn)
    dbn.pretrain(train_loader, epochs=5, learning_rate=0.1)

    # Tinh chỉnh (Dùng cả ảnh và nhãn chữ số để đánh giá)
    print("\nBắt đầu Tinh chỉnh (Fine-tuning) với Backpropagation...")
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(dbn.parameters(), lr=0.001)

    for epoch in range(10):
        epoch_loss = 0
        for data, target in train_loader:
            optimizer.zero_grad()
            outputs = dbn(data)
            loss = criterion(outputs, target)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        print(f"  Epoch tinh chỉnh {epoch+1}/10 - Loss phân loại trung bình: {epoch_loss/len(train_loader):.4f}")

Đang tải dữ liệu MNIST...


100%|██████████| 9.91M/9.91M [00:02<00:00, 3.86MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 108kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.12MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.59MB/s]


Bắt đầu Tiền huấn luyện (Pre-training) với dữ liệu MNIST...
  -> Đang huấn luyện RBM tầng 1...
     Epoch 1/5 - Sai số tái tạo trung bình: 0.0223
     Epoch 2/5 - Sai số tái tạo trung bình: 0.0146
     Epoch 3/5 - Sai số tái tạo trung bình: 0.0127
     Epoch 4/5 - Sai số tái tạo trung bình: 0.0118
     Epoch 5/5 - Sai số tái tạo trung bình: 0.0112
  -> Đang huấn luyện RBM tầng 2...
     Epoch 1/5 - Sai số tái tạo trung bình: 0.1222
     Epoch 2/5 - Sai số tái tạo trung bình: 0.1006
     Epoch 3/5 - Sai số tái tạo trung bình: 0.0963
     Epoch 4/5 - Sai số tái tạo trung bình: 0.0942
     Epoch 5/5 - Sai số tái tạo trung bình: 0.0928

Bắt đầu Tinh chỉnh (Fine-tuning) với Backpropagation...
  Epoch tinh chỉnh 1/10 - Loss phân loại trung bình: 0.4831
  Epoch tinh chỉnh 2/10 - Loss phân loại trung bình: 0.1648
  Epoch tinh chỉnh 3/10 - Loss phân loại trung bình: 0.1145
  Epoch tinh chỉnh 4/10 - Loss phân loại trung bình: 0.0839
  Epoch tinh chỉnh 5/10 - Loss phân loại trung bình: 0.0638
  E